In [ ]:
import psycopg2
import json


# Connect to db
conn = psycopg2.connect(
    dbname="congress_db",
    user="postgres",
    password="data4001",
    host="localhost",
    port="5432"
)

cur = conn.cursor()
conn.commit()

In [11]:
# Import bills
with open("../data/gold/bills_119.json", "r") as f:
    bills = json.load(f)

bill_insert_query = """
INSERT INTO bills (
    bill_id,
    bill_type,
    bill_num,
    congress,
    chamber,
    title
)
VALUES (%s, %s, %s, %s, %s, %s)
ON CONFLICT (bill_id) DO NOTHING;
"""

for b in bills:
    cur.execute(bill_insert_query, (
        b.get("bill_id"),
        b.get("bill_type"),
        b.get("bill_num"),
        b.get("congress"),
        b.get("chamber"),
        b.get("title")
    ))

conn.commit()

In [12]:
# Import amendments
with open("../data/gold/amendments_119.json", "r") as f:
    amendments = json.load(f)

amendment_insert_query = """
INSERT INTO amendments (
    amendment_id,
    amendment_type,
    congress,
    chamber,
    purpose
)
VALUES (%s, %s, %s, %s, %s)
ON CONFLICT (amendment_id) DO NOTHING;
"""

for a in amendments:
    cur.execute(amendment_insert_query, (
        a.get("amendment_id"),
        a.get("amendment_type"),
        a.get("congress"),
        a.get("chamber"),
        a.get("purpose")
    ))

conn.commit()

In [8]:
import json

# Importing members file into table
with open("../data/gold/members_119.json", "r") as f:
    members = json.load(f)

insert_query = """
INSERT INTO members (
    member_id,
    full_name,
    first_name,
    last_name,
    party,
    chamber,
    state_name,
    district,
    years_in_congress,
    age
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
ON CONFLICT (member_id) DO NOTHING;
"""

for m in members:
    cur.execute(insert_query, (
        m.get("member_id"),
        m.get("full_name"),
        m.get("first_name"),
        m.get("last_name"),
        m.get("party"),
        m.get("chamber"),
        m.get("state_name"),
        m.get("district"),
        m.get("years_in_congress"),
        m.get("age")
    ))

conn.commit()


In [ ]:
# Import votes
with open("../data/gold/votes_119.json", "r") as f:
    votes = json.load(f)

vote_insert_query = """
INSERT INTO votes (
    vote_id,
    bill_id,
    question,
    chamber,
    congress,
    session_num,
    vote_date,
    result
)
VALUES (%s, %s, %s, %s, %s, %s, %s)
ON CONFLICT (vote_id) DO NOTHING;
"""

for v in votes:
    cur.execute(vote_insert_query, (
        v.get("vote_id"),
        v.get("bill_id"),
        v.get("chamber"),
        v.get("congress"),
        v.get("session_num"),
        v.get("vote_date"),
        v.get("result")
    ))

conn.commit()

In [ ]:
# Import vote records
with open("../data/gold/vote_records_119.json", "r") as f:
    vote_records = json.load(f)

# Exclude values when vote_id is NOT in votes
# Exclude values when member_id is NOT in members
with open("../data/gold/votes_119.json", "r") as f:
    votes = json.load(f)
with open("../data/gold/members_119.json", "r") as f:
    members = json.load(f)
valid_vote_ids = {v.get("vote_id") for v in votes if v.get("vote_id")}
valid_member_ids = {m.get("member_id") for m in members if m.get("member_id")}
valid_vote_records = [
    vr for vr in vote_records
    if vr.get("vote_id") in valid_vote_ids and vr.get("member_id") in valid_member_ids
]

vote_record_insert_query = """
INSERT INTO vote_records (
    vote_id,
    member_id,
    position
)
VALUES (%s, %s, %s)
ON CONFLICT (vote_id, member_id) DO NOTHING;
"""

for vr in valid_vote_records:
    cur.execute(vote_record_insert_query, (
        vr.get("vote_id"),
        vr.get("member_id"),
        vr.get("position")
    ))

conn.commit()

In [ ]:
# Import vote records
with open("../data/gold/vote_party_totals_119.json", "r") as f:
    vote_party_totals = json.load(f)

# Exclude values when vote_id is NOT in votes
with open("../data/gold/votes_119.json", "r") as f:
    votes = json.load(f)
valid_vote_ids = {v.get("vote_id") for v in votes if v.get("vote_id")}
valid_vote_totals = [
    vt for vt in vote_party_totals
    if vt.get("vote_id") in valid_vote_ids
]

vote_insert_query = """
INSERT INTO vote_party_totals (
    vote_id,
    party,
    yes_count,
    no_count,
    present_count,
    not_voting_count,
)
VALUES (%s, %s, %s, %s, %s, %s, %s)
ON CONFLICT (vote_id) DO NOTHING;
"""

for v in vote_party_totals:
# for vt in valid_vote_totals:
    cur.execute(vote_insert_query, (
        v.get("vote_id"),
        v.get("party"),
        v.get("yes_count"),
        v.get("no_count"),
        v.get("present_count"),
        v.get("not_voting_count")
    ))

conn.commit()

In [ ]:
cur.close()
conn.close()